# Task 2 — Data Engineering

**Module:** 7006SCN Machine Learning and Big Data  
**Student:** VARSHAREDDY RAJIREDDYGARI

In [ ]:
# ── Step 1: Install PySpark ──────────────────────────────────────────────────
!pip install pyspark -q

In [ ]:
# ── Step 2: Download dataset if not already present ─────────────────────────
import urllib.request, os

DATA_URL  = "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2024-01.parquet"
DATA_PATH = "/content/fhvhv_tripdata_2024-01.parquet"

if not os.path.exists(DATA_PATH):
    print("Downloading NYC TLC HVFHV dataset (~451 MB) …")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Download complete.")
else:
    print(f"Dataset already present at {DATA_PATH}")

print(f"File size: {os.path.getsize(DATA_PATH)/1e6:.1f} MB")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, hour, dayofweek
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .appName("7006SCN_Task2") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

## 2.1 Data Ingestion with Partition Count

In [ ]:
df = spark.read.parquet(DATA_PATH)

num_partitions = df.rdd.getNumPartitions()
print(f"Partitions on load: {num_partitions}")
print(f"Total rows: {df.count():,}")

## 2.2 Preprocessing — Target Variable & Cleaning

In [ ]:
# Create binary target variable
df = df.withColumn("is_tipped", when(col("tips") > 0, 1).otherwise(0))

# 8% stratified sample (~1.5M rows) for efficient single-node processing
df_sampled = df.sample(withReplacement=False, fraction=0.08, seed=42)

# Drop rows with nulls in critical columns
df_clean = df_sampled.dropna(
    subset=["trip_miles", "trip_time", "base_passenger_fare", "PULocationID", "DOLocationID"]
)

print(f"Cleaned sample size: {df_clean.count():,}")

## 2.3 Feature Engineering

In [ ]:
# Extract temporal features from pickup_datetime
df_features = df_clean \
    .withColumn("pickup_hour", hour(col("pickup_datetime"))) \
    .withColumn("pickup_day",  dayofweek(col("pickup_datetime")))

selected_cols = [
    "PULocationID", "DOLocationID",
    "trip_miles", "trip_time", "base_passenger_fare",
    "pickup_hour", "pickup_day", "is_tipped"
]
df_final = df_features.select(selected_cols)
df_final.show(5)

## 2.4 PySpark Pipeline Construction

In [ ]:
categorical_cols = ["PULocationID", "DOLocationID", "pickup_hour", "pickup_day"]
numeric_cols     = ["trip_miles", "trip_time", "base_passenger_fare"]

stages = []

# StringIndexer + OneHotEncoder for each categorical column
for c in categorical_cols:
    si  = StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep")
    ohe = OneHotEncoder(inputCols=[c+"_idx"], outputCols=[c+"_vec"])
    stages += [si, ohe]

# Assemble + scale numeric features
num_asm = VectorAssembler(inputCols=numeric_cols, outputCol="num_raw")
scaler  = StandardScaler(inputCol="num_raw", outputCol="num_scaled", withStd=True, withMean=True)
stages += [num_asm, scaler]

# Final assembler
final_asm = VectorAssembler(
    inputCols=[c+"_vec" for c in categorical_cols] + ["num_scaled"],
    outputCol="features"
)
stages.append(final_asm)

pipeline       = Pipeline(stages=stages)
pipeline_model = pipeline.fit(df_final)
df_pipeline    = pipeline_model.transform(df_final)

print("Pipeline complete.")
df_pipeline.select("features", "is_tipped").show(5, truncate=True)

In [ ]:
# Save processed data for Tasks 3–6
import os
os.makedirs("/content/outputs", exist_ok=True)

df_pipeline.select("features", "is_tipped") \
    .write.mode("overwrite") \
    .parquet("/content/processed_data.parquet")

print("Processed data saved to /content/processed_data.parquet")